In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [1]:
from pyspark.sql import functions as F
df_ad_events_raw = spark.table("lh_bronze_game.ad_events_raw")

StatementMeta(, f65223d1-b5ab-4660-a855-96491a7707d1, 3, Finished, Available, Finished, False)

In [2]:
print("Row count:", df_ad_events_raw.count())
print("Column count:", len(df_ad_events_raw.columns))
df_ad_events_raw.printSchema()
display(df_ad_events_raw.limit(5))

StatementMeta(, f65223d1-b5ab-4660-a855-96491a7707d1, 4, Finished, Available, Finished, False)

Row count: 233037
Column count: 7
root
 |-- ad_type: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- placement: string (nullable = true)
 |-- player_id: string (nullable = true)
 |-- revenue_usd: double (nullable = true)
 |-- session_id: string (nullable = true)
 |-- timestamp: string (nullable = true)



SynapseWidget(Synapse.DataFrame, a625c46e-3604-4c02-a6e2-9195a8df2c99)

In [3]:
df_ad_events_raw.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_ad_events_raw.columns
]).show(truncate=False)

StatementMeta(, f65223d1-b5ab-4660-a855-96491a7707d1, 5, Finished, Available, Finished, False)

+-------+--------+---------+---------+-----------+----------+---------+
|ad_type|event_id|placement|player_id|revenue_usd|session_id|timestamp|
+-------+--------+---------+---------+-----------+----------+---------+
|0      |0       |0        |0        |306        |0         |0        |
+-------+--------+---------+---------+-----------+----------+---------+



In [4]:
df_ad_events_raw.filter(
    F.col("revenue_usd").isNull()
).groupBy("ad_type").count().show()

StatementMeta(, f65223d1-b5ab-4660-a855-96491a7707d1, 6, Finished, Available, Finished, False)

+------------+-----+
|     ad_type|count|
+------------+-----+
|interstitial|  148|
|    rewarded|  158|
+------------+-----+



In [5]:
df_ad_events_raw.filter(
    F.col("revenue_usd").isNull()
).groupBy("placement").count().show()

StatementMeta(, f65223d1-b5ab-4660-a855-96491a7707d1, 7, Finished, Available, Finished, False)

+------------+-----+
|   placement|count|
+------------+-----+
| extra_moves|   66|
|   level_end|   69|
|bonus_reward|   57|
|      revive|   54|
| home_screen|   60|
+------------+-----+



In [6]:
total_events = df_ad_events_raw.count()

unique_event_ids = (
    df_ad_events_raw
    .select("event_id")
    .distinct()
    .count()
)

print("Total events:", total_events)
print("Unique event_id:", unique_event_ids)
print("Duplicate:", total_events - unique_event_ids)

StatementMeta(, f65223d1-b5ab-4660-a855-96491a7707d1, 8, Finished, Available, Finished, False)

Total events: 233037
Unique event_id: 232744
Duplicate: 293


In [7]:
df_ad_events_clean = df_ad_events_raw.dropDuplicates()

StatementMeta(, f65223d1-b5ab-4660-a855-96491a7707d1, 9, Finished, Available, Finished, False)

In [8]:
print("Raw:", df_ad_events_raw.count())
print("Clean:", df_ad_events_clean.count())

StatementMeta(, f65223d1-b5ab-4660-a855-96491a7707d1, 10, Finished, Available, Finished, False)

Raw: 233037
Clean: 232744


In [9]:
df_ad_check = (
    df_ad_events_clean
    .withColumn(
        "timestamp_parsed",
        F.to_timestamp("timestamp")
    )
)

df_ad_check.select(
    F.count(
        F.when(F.col("timestamp_parsed").isNull(), 1)
    ).alias("invalid_timestamp")
).show()

StatementMeta(, f65223d1-b5ab-4660-a855-96491a7707d1, 11, Finished, Available, Finished, False)

+-----------------+
|invalid_timestamp|
+-----------------+
|              312|
+-----------------+



In [10]:
display(
    df_ad_events_clean
    .filter(
        F.to_timestamp("timestamp").isNull()
    )
    .select(
        "event_id",
        "player_id",
        "session_id",
        "timestamp"
    )
    .limit(20)
)

StatementMeta(, f65223d1-b5ab-4660-a855-96491a7707d1, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b7aaa763-8d90-4427-9fec-12e5dd74791e)

In [11]:
df_ad_events_clean = (
    df_ad_events_clean
    .filter(F.to_timestamp("timestamp").isNotNull())
    .withColumn(
        "timestamp",
        F.to_timestamp("timestamp")
    )
)

StatementMeta(, f65223d1-b5ab-4660-a855-96491a7707d1, 13, Finished, Available, Finished, False)

In [12]:
df_ad_events_clean.select(
    F.min("revenue_usd").alias("min_revenue"),
    F.max("revenue_usd").alias("max_revenue"),
    F.avg("revenue_usd").alias("avg_revenue")
).show()

StatementMeta(, f65223d1-b5ab-4660-a855-96491a7707d1, 14, Finished, Available, Finished, False)

+-----------+-----------+--------------------+
|min_revenue|max_revenue|         avg_revenue|
+-----------+-----------+--------------------+
|      0.003|      0.045|0.019756583062646996|
+-----------+-----------+--------------------+



In [13]:
df_ad_events_clean.select(
    F.min("revenue_usd").alias("min_revenue"),
    F.max("revenue_usd").alias("max_revenue"),
    F.avg("revenue_usd").alias("avg_revenue")
).show()

StatementMeta(, f65223d1-b5ab-4660-a855-96491a7707d1, 15, Finished, Available, Finished, False)

+-----------+-----------+--------------------+
|min_revenue|max_revenue|         avg_revenue|
+-----------+-----------+--------------------+
|      0.003|      0.045|0.019756583062646996|
+-----------+-----------+--------------------+



In [14]:
orphan_ad_players = (
    df_ad_events_clean.alias("a")
    .join(
        spark.table("lh_silver_game.players_clean").alias("p"),
        F.col("a.player_id") == F.col("p.player_id"),
        "left_anti"
    )
)

print("Players tablosunda bulunmayan ad event:", orphan_ad_players.count())

StatementMeta(, f65223d1-b5ab-4660-a855-96491a7707d1, 16, Finished, Available, Finished, False)

Players tablosunda bulunmayan ad event: 0


In [15]:
print("Final row count:", df_ad_events_clean.count())

df_ad_events_clean.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in [
        "event_id",
        "player_id",
        "session_id",
        "ad_type",
        "placement",
        "timestamp"
    ]
]).show()

df_ad_events_clean.printSchema()

StatementMeta(, f65223d1-b5ab-4660-a855-96491a7707d1, 17, Finished, Available, Finished, False)

Final row count: 232432
+--------+---------+----------+-------+---------+---------+
|event_id|player_id|session_id|ad_type|placement|timestamp|
+--------+---------+----------+-------+---------+---------+
|       0|        0|         0|      0|        0|        0|
+--------+---------+----------+-------+---------+---------+

root
 |-- ad_type: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- placement: string (nullable = true)
 |-- player_id: string (nullable = true)
 |-- revenue_usd: double (nullable = true)
 |-- session_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [16]:
df_ad_events_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lh_silver_game.ad_events_clean")

StatementMeta(, f65223d1-b5ab-4660-a855-96491a7707d1, 18, Finished, Available, Finished, False)

In [17]:
df_check = spark.table("lh_silver_game.ad_events_clean")

print("Saved row count:", df_check.count())
display(df_check.limit(5))

StatementMeta(, f65223d1-b5ab-4660-a855-96491a7707d1, 19, Finished, Available, Finished, False)

Saved row count: 232432


SynapseWidget(Synapse.DataFrame, 2a4fa6c3-497a-4095-bfd5-3e9eb575cf5d)